<a href="https://colab.research.google.com/github/tzs69/BT4012-kaggle-comp/blob/main/BT4012_competition_baseline_modelling_unlabelled_ir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unlabelled Data Experiments

Illicit vs rest modelling notebook where I train XGBoost and LightGBM on both labelled and unlabelled data, then save their models and preds for use inside the final ensemble modelling notebook.

Illicit-versus-rest classifier:
- Known illicit rows receive `target = 1`
- Known licit and unlabelled rows receive `target = 0`

Unknown rows may contain illicit transactions, but due to extreme skewed proportion of licit vs illicit transactions, this target is an experimental assumption I am willing to make

Prerequisites: Run the supervised baseline notebook first to create `base_models_cache/xgb.joblib` and `base_models_cache/lgbm.joblib`

In [1]:
%pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 32.9 MB/s  0:00:03


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive
from joblib import load, dump
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from types import SimpleNamespace

In [5]:
drive.mount("/content/drive")

ROOT_DIR = "/content/drive/MyDrive/BT4012_competition"
df_train = pd.read_csv(ROOT_DIR + "/transformed_datasets/train_topo.csv")
df_test = pd.read_csv(ROOT_DIR + "/transformed_datasets/test_topo.csv")

CACHE_DIR = ROOT_DIR + "/base_models_cache"
SUBMISSIONS_DIR = ROOT_DIR + "/submissions"

sample_submission = pd.read_csv(ROOT_DIR + "/base_datasets/sample_submission.csv")

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
Path(SUBMISSIONS_DIR).mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## Load Saved Model Bundles and Predictions
Load the supervised bundles together, same as inside final ensemble modelling. Keep their fitted models as well, since XGBoost / CatBoost IR reuse those configs and each model uses its own fitted feature columns.

In [ ]:
cached = {}
for name in ["xgb", "lgbm", "catboost"]:
    path = Path(CACHE_DIR + f"/{name}.joblib")
    if not path.exists():
        continue

    bundle = load(path)
    cached[name] = {
        "model": bundle["model"],
        "oof": bundle["oof"].set_index("txId"),
        "test": bundle["test_predictions"].set_index("txId"),
        "feature_cols": bundle.get("feature_cols", []),
    }
del bundle

loaded_models = ", ".join(cached)
print(f"Loaded: {loaded_models}")

Loaded: xgb, lgbm, catboost


## Reusable Data


###Illicit vs Rest Target
Keep all train rows, incl. unlabelled txns
- Only known illicit txns `target = 1`
- Everything else `target = 0`

In [ ]:
df_train["label"] = pd.to_numeric(df_train["label"], errors="coerce")
known = df_train["label"].notna()
y_ir = df_train["label"].eq(1).astype(int)

display(df_train["label"].value_counts(dropna=False))
print(f"Known-label txns: {known.sum():,} | Unlabelled txns: {(~known).sum():,}")

,count
label,
NaN,110537
0.0,27591
1.0,3644


Known-label txns: 31,235 | Unlabelled txns: 110,537


### Temporal Fold Train/Val Splits


In [ ]:
fold_windows = [
    (20, 21, 25),
    (25, 26, 30),
    (30, 31, 35),
]

times = df_train["time_step"]
cv_splits = [
    (
        np.flatnonzero(times.le(train_end)),
        np.flatnonzero(times.between(val_start, val_end) & known),
    )
    for train_end, val_start, val_end in fold_windows
]


## Helper Functions


### Grid Search Cross Validation

In [ ]:
def run_gscv(model, param_grid, X, y, splits):
    search = GridSearchCV(
        model,
        param_grid,
        cv=splits,
        scoring="roc_auc",
        n_jobs=1,
        refit=False,
        return_train_score=False,
    )
    return search.fit(X, y)

### Temporal Folds Validation
For each fold:
 - Fit a fresh model on train rows and get preds for val rows
 - Save OOF preds and calculate ROC-AUC / AP

In [ ]:
def evaluate_temporal(model, X, y, splits):
    scores = []
    oof = pd.Series(np.nan, index=X.index, name="prediction")

    for fold, (tr, va) in enumerate(splits, start=1):
        fitted = clone(model).fit(X.iloc[tr], y.iloc[tr])
        pred = fitted.predict_proba(X.iloc[va])[:, 1]
        oof.iloc[va] = pred

        scores.append({
            "fold": fold,
            "n_train": len(tr),
            "n_val": len(va),
            "auc": roc_auc_score(y.iloc[va], pred),
            "ap": average_precision_score(y.iloc[va], pred),
        })

    return pd.DataFrame(scores), oof

### Run Predictions


In [ ]:
def fit_predict(model, X, y, X_test):
    fitted = clone(model).fit(X, y)
    return fitted, fitted.predict_proba(X_test)[:, 1]


### Persist Run Data
Save the fitted model, feature columns, oof / test preds and temporal scores into `base_models_cache/{name}.joblib`.

Only known-label txn rows are kept and saved in the oof table (so same as the supervised baseline model oof preds). Rows before the validation windows keep `fold = -1` and no oof pred.

In [ ]:
def save_ir_cache(name, model, cols, scores, oof, test_pred):
    # Assign validation fold numbers; earlier train-only rows stay -1
    fold_ids = np.full(len(df_train), -1, dtype=int)
    for fold, (_, va) in enumerate(cv_splits, start=1):
        fold_ids[va] = fold

    # Keep known-label rows only for scoring / blending with supervised oof preds
    oof_table = df_train.loc[known, ["txId", "time_step", "label"]].copy()
    oof_table["label"] = oof_table["label"].astype(int)
    oof_table["fold"] = fold_ids[known.to_numpy()]
    oof_table["prediction"] = oof.loc[known].to_numpy()

    # Test preds matched to their assoc. txn IDs
    test_table = df_test[["txId", "time_step"]].copy()
    test_table["prediction"] = np.asarray(test_pred)

    bundle = {
        "model": model,
        "feature_cols": list(cols),
        "oof": oof_table.reset_index(drop=True),
        "test_predictions": test_table,
        "cv_scores": scores,
    }

    cache_path = CACHE_DIR + f"/{name}.joblib"
    dump(bundle, cache_path)
    print(f"Saved {name}: {len(cols)} features -> {cache_path}")

## XGBoost


### Define XGBoost Model / Choose Feature Columns
Use same feature cols as used by the supervised baseline models trained on only known-label rows

In [ ]:
xgb_ir_cols = list(cached["xgb"]["model"].feature_names_in_)

# Fix sampling / regularisation settings for this experiment
xgb_ir_base = Pipeline([
    ("xgb", XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="auc",
        min_child_weight=1.0,
        gamma=0.0,
        subsample=0.8,
        colsample_bytree=0.5,
        reg_alpha=0.0,
        reg_lambda=1.0,
        scale_pos_weight=1.0,
        random_state=42,
        n_jobs=2,
    )),
])

print(f"Features: {len(xgb_ir_cols)}")
display(xgb_ir_base)

Features: 165


Pipeline(steps=[('xgb',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.5, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=True, eval_metric='auc',
                               feature_types=None, feature_weights=None,
                               gamma=0.0, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=1.0, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=2,
                               num_parallel_tree=None, ...))])

### XGBoost GSCV

In [ ]:
# Tune learning rate / tree count first; keep other params fixed
xgb_ir_param_grid = {
    "xgb__learning_rate": [0.02, 0.05],
    "xgb__n_estimators": [200, 600, 1000],
    "xgb__max_depth": [4, 6],
}

xgb_ir_search = run_gscv(
    xgb_ir_base,
    xgb_ir_param_grid,
    df_train[xgb_ir_cols],
    y_ir,
    cv_splits,
)

print(f"Best params: {xgb_ir_search.best_params_}")
print(f"Best mean AUC: {xgb_ir_search.best_score_}")

Best params: {'xgb__learning_rate': 0.05, 'xgb__max_depth': 4, 'xgb__n_estimators': 1000}
Best mean AUC: 0.9503795115816485


### XGBoost Params Tuning


In [ ]:
xgb_ir_search = SimpleNamespace(
    best_params_={
        "xgb__learning_rate": 0.05,
        "xgb__max_depth": 4,
        "xgb__n_estimators": 1000
    }
)

xgb_ir_model = clone(xgb_ir_base).set_params(
    **xgb_ir_search.best_params_
)

### XGBoost Temporal Folds Validation


In [ ]:
xgb_ir_scores, xgb_ir_oof = evaluate_temporal(
    xgb_ir_model,
    df_train[xgb_ir_cols],
    y_ir,
    cv_splits,
)

display(xgb_ir_scores)
mean_auc = xgb_ir_scores["auc"].mean()
print(f"Mean AUC: {mean_auc}")

,fold,n_train,n_val,auc,ap
0,1,90762,5311,0.862541,0.833131
1,2,111264,2705,0.989797,0.974305
2,3,123287,4330,0.998800,0.994668


Mean AUC: 0.9503795115816485


### Supervised XGBoost Comparison
Compare supervised vs ir scores on the exact same val txns.

Standalone ir scores do not tell us whether it helps the ensemble, that one test inside final modelling.


In [ ]:
supervised_oof = cached["xgb"]["oof"]
comparison_rows = []

for fold, (_, va) in enumerate(cv_splits, start=1):
    reference = supervised_oof.loc[df_train.iloc[va]["txId"]]
    # Cached supervised scores must cover these same validation folds.
    assert reference["fold"].eq(fold).all(), "Regenerate the supervised cache using the same folds."

    target = df_train.iloc[va]["label"].astype(int)
    comparison_rows.append({
        "fold": fold,
        "supervised_auc": roc_auc_score(target, reference["prediction"]),
        "ir_auc": xgb_ir_scores.iloc[fold - 1]["auc"],
        "supervised_ap": average_precision_score(target, reference["prediction"]),
        "ir_ap": xgb_ir_scores.iloc[fold - 1]["ap"],
    })

comparison = pd.DataFrame(comparison_rows)
comparison["auc_change"] = comparison["ir_auc"] - comparison["supervised_auc"]
display(comparison)

,fold,supervised_auc,ir_auc,supervised_ap,ir_ap,auc_change
0,1,0.959041,0.862541,0.876202,0.833131,-0.096500
1,2,0.994148,0.989797,0.983568,0.974305,-0.004351
2,3,0.997770,0.998800,0.989623,0.994668,0.001030


### XGBoost Predictions / Persist Run Data


In [ ]:
xgb_ir_final, xgb_ir_test_pred = fit_predict(
    xgb_ir_model,
    df_train[xgb_ir_cols],
    y_ir,
    df_test[xgb_ir_cols],
)


save_ir_cache(
    "xgb_ir",
    xgb_ir_final,
    xgb_ir_cols,
    xgb_ir_scores,
    xgb_ir_oof,
    xgb_ir_test_pred,
)

Saved xgb_ir: 165 features -> /content/drive/MyDrive/BT4012_competition/base_models_cache/xgb_ir.joblib


## LightGBM

###Define LightGBM Model / Choose Feature Columns

In [ ]:
lgbm_ir_cols = list(cached["lgbm"]["model"].feature_names_in_)

lgbm_ir_base = Pipeline([
    ("lgbm", LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.5,
        reg_alpha=0.0,
        reg_lambda=0.0,
        scale_pos_weight=1.0,
        random_state=42,
        n_jobs=2,
        deterministic=True,
        force_row_wise=True,
        verbosity=-1,
    )),
])

print(f"Features: {len(lgbm_ir_cols)}")
display(lgbm_ir_base)

Features: 165


Pipeline(steps=[('lgbm',
                 LGBMClassifier(colsample_bytree=0.5, deterministic=True,
                                force_row_wise=True, n_jobs=2,
                                objective='binary', random_state=42,
                                scale_pos_weight=1.0, subsample=0.8,
                                subsample_freq=1, verbosity=-1))])

###LightGBM GSCV

In [ ]:
# Search around the slower boosting config
lgbm_ir_param_grid = {
    "lgbm__learning_rate": [0.02, 0.05],
    "lgbm__n_estimators": [500, 1000, 1500],
    "lgbm__num_leaves": [15, 31],
}

lgbm_ir_search = run_gscv(
    lgbm_ir_base,
    lgbm_ir_param_grid,
    df_train[lgbm_ir_cols],
    y_ir,
    cv_splits,
)

print(f"Best params: {lgbm_ir_search.best_params_}")
print(f"Best mean AUC: {lgbm_ir_search.best_score_}")

Best params: {'lgbm__learning_rate': 0.05, 'lgbm__n_estimators': 1000, 'lgbm__num_leaves': 31}
Best mean AUC: 0.9608444301671666


###LightGBM Params Tuning

In [ ]:
lgbm_ir_search = SimpleNamespace(
    best_params_={
        "lgbm__learning_rate": 0.05,
        "lgbm__n_estimators": 1000,
        "lgbm__num_leaves": 31
    }
)

lgbm_ir_model = clone(lgbm_ir_base).set_params(
    **lgbm_ir_search.best_params_
)

###LightGBM Temporal Folds Validation

In [ ]:
lgbm_ir_scores, lgbm_ir_oof = evaluate_temporal(
    lgbm_ir_model,
    df_train[lgbm_ir_cols],
    y_ir,
    cv_splits,
)

display(lgbm_ir_scores)
mean_auc = lgbm_ir_scores["auc"].mean()
print(f"Mean AUC: {mean_auc}")

,fold,n_train,n_val,auc,ap
0,1,90762,5311,0.891695,0.838604
1,2,111264,2705,0.992476,0.981073
2,3,123287,4330,0.998362,0.992840


Mean AUC: 0.9608444301671666


###Supervised LightGBM Comparison

In [ ]:
supervised_oof = cached["lgbm"]["oof"]
comparison_rows = []

for fold, (_, va) in enumerate(cv_splits, start=1):
    reference = supervised_oof.loc[df_train.iloc[va]["txId"]]
    # Cached supervised scores must cover these same validation folds.
    assert reference["fold"].eq(fold).all(), "Regenerate the supervised cache using the same folds."

    target = df_train.iloc[va]["label"].astype(int)
    comparison_rows.append({
        "fold": fold,
        "supervised_auc": roc_auc_score(target, reference["prediction"]),
        "ir_auc": lgbm_ir_scores.iloc[fold - 1]["auc"],
        "supervised_ap": average_precision_score(target, reference["prediction"]),
        "ir_ap": lgbm_ir_scores.iloc[fold - 1]["ap"],
    })

comparison = pd.DataFrame(comparison_rows)
comparison["auc_change"] = comparison["ir_auc"] - comparison["supervised_auc"]
display(comparison)

,fold,supervised_auc,ir_auc,supervised_ap,ir_ap,auc_change
0,1,0.946823,0.891695,0.868312,0.838604,-0.055128
1,2,0.994499,0.992476,0.984509,0.981073,-0.002023
2,3,0.997826,0.998362,0.990170,0.992840,0.000536


###LightGBM Predictions / Persist Run Data

In [ ]:
lgbm_ir_final, lgbm_ir_test_pred = fit_predict(
    lgbm_ir_model,
    df_train[lgbm_ir_cols],
    y_ir,
    df_test[lgbm_ir_cols],
)

save_ir_cache(
    "lgbm_ir",
    lgbm_ir_final,
    lgbm_ir_cols,
    lgbm_ir_scores,
    lgbm_ir_oof,
    lgbm_ir_test_pred,
)

Saved lgbm_ir: 165 features -> /content/drive/MyDrive/BT4012_competition/base_models_cache/lgbm_ir.joblib


##CatBoost

### Define CatBoost Model / Choose Feature Columns

In [ ]:
catboost_ir_cols = list(cached["catboost"]["feature_cols"])

# Fix regularisation / training settings for this experiment
catboost_ir_base = Pipeline([
    ("catboost", CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        l2_leaf_reg=3.0,
        random_seed=42,
        thread_count=2,
        task_type="CPU",
        boosting_type="Plain",
        use_best_model=False,
        verbose=False,
        allow_writing_files=False,
    )),
])

print(f"Features: {len(catboost_ir_cols)}")
display(catboost_ir_base)

Features: 165


Pipeline(steps=[('catboost',
                 CatBoostClassifier(allow_writing_files=False, boosting_type='Plain', eval_metric='AUC', l2_leaf_reg=3.0, loss_function='Logloss', random_seed=42, task_type='CPU', thread_count=2, use_best_model=False, verbose=False))])

###CatBoost Params Tuning

In [ ]:
# Search boosting budget and tree depth; keep other params fixed
catboost_ir_param_grid = {
    "catboost__learning_rate": [0.02, 0.05],
    "catboost__iterations": [500, 1000, 1500],
    "catboost__depth": [4, 6],
}

catboost_ir_search = run_gscv(
    catboost_ir_base,
    catboost_ir_param_grid,
    df_train[catboost_ir_cols],
    y_ir,
    cv_splits,
)

print(f"Best params: {catboost_ir_search.best_params_}")
print(f"Best mean AUC: {catboost_ir_search.best_score_}")

Best params: {'catboost__depth': 4, 'catboost__iterations': 500, 'catboost__learning_rate': 0.05}
Best mean AUC: 0.9485898637638205


In [ ]:
catboost_ir_search = SimpleNamespace(
    best_params_={
        "catboost__depth": 4,
        "catboost__iterations": 500,
        "catboost__learning_rate": 0.05
    }
)

catboost_ir_model = clone(catboost_ir_base).set_params(
    **catboost_ir_search.best_params_
)

### CatBoost Temporal Folds Validation

In [ ]:
catboost_ir_scores, catboost_ir_oof = evaluate_temporal(
    catboost_ir_model,
    df_train[catboost_ir_cols],
    y_ir,
    cv_splits,
)

display(catboost_ir_scores)
mean_auc = catboost_ir_scores["auc"].mean()
print(f"Mean AUC: {mean_auc}")

,fold,n_train,n_val,auc,ap
0,1,90762,5311,0.856432,0.809796
1,2,111264,2705,0.992999,0.981685
2,3,123287,4330,0.996339,0.987024


Mean AUC: 0.9485898637638205


### Supervised CatBoost Comparison

In [ ]:
supervised_oof = cached["catboost"]["oof"]
comparison_rows = []

for fold, (_, va) in enumerate(cv_splits, start=1):
    reference = supervised_oof.loc[df_train.iloc[va]["txId"]]
    # Cached supervised scores must cover these same validation folds.
    assert reference["fold"].eq(fold).all(), "Regenerate the supervised cache using the same folds."

    target = df_train.iloc[va]["label"].astype(int)
    comparison_rows.append({
        "fold": fold,
        "supervised_auc": roc_auc_score(target, reference["prediction"]),
        "ir_auc": catboost_ir_scores.iloc[fold - 1]["auc"],
        "supervised_ap": average_precision_score(target, reference["prediction"]),
        "ir_ap": catboost_ir_scores.iloc[fold - 1]["ap"],
    })

comparison = pd.DataFrame(comparison_rows)
comparison["auc_change"] = comparison["ir_auc"] - comparison["supervised_auc"]
display(comparison)

,fold,supervised_auc,ir_auc,supervised_ap,ir_ap,auc_change
0,1,0.898759,0.856432,0.850627,0.809796,-0.042326
1,2,0.994868,0.992999,0.987200,0.981685,-0.001870
2,3,0.998507,0.996339,0.992028,0.987024,-0.002168


###CatBoost Predictions / Persist Run Data

In [ ]:
catboost_ir_final, catboost_ir_test_pred = fit_predict(
    catboost_ir_model,
    df_train[catboost_ir_cols],
    y_ir,
    df_test[catboost_ir_cols],
)


save_ir_cache(
    "catboost_ir",
    catboost_ir_final,
    catboost_ir_cols,
    catboost_ir_scores,
    catboost_ir_oof,
    catboost_ir_test_pred,
)

Saved catboost_ir: 165 features -> /content/drive/MyDrive/BT4012_competition/base_models_cache/catboost_ir.joblib
